# Radioactive Decay Integrators and Python Basics

Plotting and animation, Euler stability and error, and comparisons with second- and fourth-order Runge–Kutta methods.

Computational physics coursework by Udayan Sharma. Original code comments and saved outputs are retained. See the repository README for execution notes.


In [ ]:
# Task 1c - Plotting e^(-x/3) * sin(5x)

import numpy as np;
import matplotlib.pyplot as plt;


x_values = np.linspace(0, 4 * np.pi, 1000);
y_values = np.exp(-x_values/3) * np.sin(5*x_values);

# Asked AI for syntax for plotting because it has been 4 years since I last took a python class
plt.figure(figsize=(10, 6));
plt.plot(x_values, y_values, color='b', label=r'$y = e^{-x/3}\sin(5x)$');


plt.title(r'Plot of $e^{-x/3}\sin(5x)$ vs x');
plt.xlabel('x values');
plt.ylabel(r'$e^{-x/3}\sin(5x)$ values');
plt.legend();
plt.grid(True);
plt.xlim(0, 4 * np.pi);
plt.show();


In [ ]:
# Task 1d - Textfile

file_path = '/number.txt';
sum = 0;

with open(file_path , 'r') as f:
  content = f.read();
  numbers = content.split();
  for number in numbers:
    sum += int(number);
  print(content);

  print(f"The sum of the numbers in the file is: {sum}");

In [ ]:
# Task 1e - 3 Variable Animation

import matplotlib.pyplot as plt;
import numpy as np;
import matplotlib.animation as animation;

# I had absolutely no idea how to make a movie animation so I used AI for this stuff given my main code
from IPython.display import HTML;
plt.rcParams["animation.html"] = "jshtml";

N = 200;
Nt = 120;

# Creating and populating arrays
x = np.linspace(0, 1, N);
y = np.linspace(0, 1, N);
X, Y = np.meshgrid(x, y);

t_values = np.linspace(0, 1, Nt, endpoint=False);

# Forming figure out line
fig, ax = plt.subplots(figsize=(6, 5));
ax.set_xlabel("x");
ax.set_ylabel("y");
ax.set_title(r"$f(x,y,t)=\sin(2\pi x)\sin(2\pi y)\sin(2\pi t)$");

# Instantiating inital values
t0 = t_values[0];
Z0 = np.sin(2*np.pi*X) * np.sin(2*np.pi*Y) * np.sin(2*np.pi*t0);

# Used AI to learn how to format function on the plot for the range [-1,1]
im = ax.imshow(Z0, origin="lower", extent=[0, 1, 0, 1], vmin = -1, vmax = 1);
plt.colorbar(im, ax = ax, label="f");

time_text = ax.text(0.02, 1.02, "", transform=ax.transAxes);

# Self described function - updates frame from 0.02 to 1.02 "seconds" for f(x,y,t) or z in this case
def frameUpdater(i):
    t = t_values[i];
    Z = np.sin(2 * np.pi * X) * np.sin(2 * np.pi * Y) * np.sin(2 * np.pi * t);
    im.set_data(Z);
    time_text.set_text(f"t = {t:.3f}");
    return im, time_text;

ani = animation.FuncAnimation(fig, frameUpdater, frames = len(t_values), interval = 30, blit = True);

# We dont use plot.show() here since we want an animating frame
HTML(ani.to_jshtml());

# EOF



In [ ]:
# Task 2 - Euler Approximation for Nuclear Decay
#
# for PHYS 580, Lab 1, Spring 2026
#
# Uses Euler's method to solve dN/dt = -N/tau, with initial condition N(t=0) = N0


from math import exp;    # need exp()
import numpy as np;

import matplotlib;

# Simply putting matplotlib.use for TkAgg was breaking randomly so I just put a basic pass exception
try:
    matplotlib.use('TkAgg');
except Exception:
    pass

import matplotlib.pyplot as plt;


# Parameters
N0  = 1000;           # initial number of atoms
tau = 1.0;            # mean lifetime - set to 1, i.e., measure time in units of tau
tmax = 5.0 * tau;     # integrate from tau = 0 to 5

# Defning different dt/tau values to be used as specified by the manual
dt_over_tau_list = [0.05, 0.2, 0.8, 1.0, 1.5];

# Creating exact solution on a finer grid for a smooth curve with super small t
t_exact = np.linspace(0.0, tmax, 1000);
N_exact = np.array([N0 * exp(-t / tau) for t in t_exact]);

plt.figure();
plt.xlabel('t / tau');
plt.ylabel('N(t)');
plt.title('Euler approximation vs exact solution');

# Ploting exact solution first
plt.plot(t_exact / tau, N_exact, label = "exact", linewidth = 2);

# Looping over each dt value for its plot
for dt_over_tau in dt_over_tau_list:

    dt = dt_over_tau * tau;

    # I am building a time grid from 0 to tmax, using steps of dt, and if dt does not land exactly on tmax, we take a final shorter step to reach tmax.
    times_list = [0.0];

    # This is really only used for the 0.8 and 1.5 value of dt/tau
    while times_list[-1] + dt < tmax:
        times_list.append(times_list[-1] + dt);

    if times_list[-1] < tmax:
        times_list.append(tmax);

    times = np.array(times_list);

    nsteps = len(times) - 1;

    # Storage for Euler solution
    Natoms = np.zeros(nsteps + 1);
    Natoms[0] = N0;

    # Euler update timestep by timestep (allow final step smaller than dt if needed)
    for i in range(nsteps):
        dt_step = times[i + 1] - times[i];
        Natoms[i + 1] = Natoms[i] - Natoms[i] * dt_step / tau;

    # Plot Euler solution for this particular dt/tau
    plt.plot(times / tau, Natoms, marker = 'o', markersize = 3, label = f"Euler, dt/tau = {dt_over_tau:g}");


plt.legend();
plt.grid(True, alpha = 0.3);
plt.show();

#EOF



In [ ]:
# Task 3 - Deviations & Errors in Euler Approximation for Nuclear Decay
#
# for PHYS 580, Lab 1, Spring 2026
#
# Uses Euler's method to solve dN/dt = -N/tau, with initial condition N(t=0) = N0

from math import exp;    # need exp()
import numpy as np;

import matplotlib;

# Simply putting matplotlib.use for TkAgg was breaking randomly so I just put a basic pass exception
try:
    matplotlib.use('TkAgg');
except Exception:
    pass

import matplotlib.pyplot as plt;


# Parameters
N0  = 1000;           # initial number of atoms
tau = 1.0;            # mean lifetime - set to 1, i.e., measure time in units of tau
tmax = 5.0 * tau;     # integrate from tau = 0 to 5

# Defning different dt/tau values to be used as specified by the lab manual
dt_over_tau_list = [0.05, 0.2, 0.8, 1.0, 1.5];


# Lists to store results for each dt/tau
all_times = [];
all_abs_error = [];
all_frac_error = [];

fixed_abs_errors = [];     # errors at t = 5 tau
fixed_frac_errors = [];    # fractional errors at t = 5 tau


# Looping over each dt value for its plot and errors
for dt_over_tau in dt_over_tau_list:

    dt = dt_over_tau * tau;

    # I am building a time grid from 0 to tmax, using steps of dt, and if dt does not land exactly on tmax, we take a final shorter step to reach tmax.
    times_list = [0.0];

    while times_list[-1] + dt < tmax:
        times_list.append(times_list[-1] + dt);

    if times_list[-1] < tmax:
        times_list.append(tmax);

    times = np.array(times_list);
    steps = len(times) - 1;

    # Storage for Euler solution
    Natoms = np.zeros(steps + 1);
    Natoms[0] = N0;

    # Euler update timestep by timestep (allow final step smaller than dt if needed)
    for i in range(steps):
        dt_step = times[i + 1] - times[i];
        Natoms[i + 1] = Natoms[i] - Natoms[i] * dt_step / tau;

    # Exact solution on SAME times
    Nexact = np.array([N0 * exp(-t / tau) for t in times]);

    # Global deviation at each time
    error = Natoms - Nexact;
    abs_error = np.abs(error);
    frac_error = abs_error / np.abs(Nexact);

    # Storring error values in corresponding arrays for later plotting
    all_times.append(times);
    all_abs_error.append(abs_error);
    all_frac_error.append(frac_error);

    # Storing errors at t = 5 * tau (last point is tmax)
    fixed_abs_errors.append(abs_error[-1]);
    fixed_frac_errors.append(frac_error[-1]);


# Absolute global error vs time
plt.figure();
plt.xlabel('t / tau');
plt.ylabel(r'|N_Euler - N_exact|');
plt.title('Global deviation: absolute error vs time');

# Simply iterating through the errors array and plotting them
for k in range(len(dt_over_tau_list)):
    dt_over_tau = dt_over_tau_list[k];
    plt.plot(all_times[k] / tau, all_abs_error[k], marker = 'o', markersize = 3, label = f"dt/tau = {dt_over_tau:g}");

plt.legend();
plt.grid(True, alpha = 0.3);
plt.show();

# Fractional global error vs time

plt.figure();
plt.xlabel('t / tau');
plt.ylabel(r'|N_Euler - N_exact| / |N_exact|');
plt.title('Global deviation: fractional error vs time');

# Same as above
for k in range(len(dt_over_tau_list)):
    dt_over_tau = dt_over_tau_list[k];
    plt.plot(all_times[k] / tau, all_frac_error[k], marker = 'o', markersize = 3, label = f"dt/tau = {dt_over_tau:g}");

plt.legend();
plt.grid(True, alpha = 0.3);
plt.yscale('log');   # useful because errors can span orders of magnitude
plt.show();

# Error vs step size at fixed time (t = 5 tau)
plt.figure();
plt.xlabel('dt / tau');
plt.ylabel(r'|error| at t = 5 tau');
plt.title('Step-size dependence at fixed time (t = 5 tau)');
plt.plot(dt_over_tau_list, fixed_abs_errors, marker = 'o');
plt.grid(True, alpha = 0.3);
plt.show();

plt.figure();
plt.xlabel('dt / tau');
plt.ylabel(r'|error| / |exact| at t = 5 tau');
plt.title('Fractional error vs step size at fixed time (t = 5 tau)');
plt.plot(dt_over_tau_list, fixed_frac_errors, marker = 'o');
plt.grid(True, alpha = 0.3);
plt.show();

#EOF



In [ ]:
# Task 4 - Runge-Kutta 2nd & 4th order approximations with errors and comparison to Euler and exact solution
#
# For PHYS 580, Lab 1, Spring 2026

from math import exp;    # need exp()
import numpy as np;

import matplotlib;

# Simply putting matplotlib.use for TkAgg was breaking randomly so I just put a basic pass exception
try:
    matplotlib.use('TkAgg');
except Exception:
    pass

import matplotlib.pyplot as plt;


# Parameters
N0  = 1000;           # initial number of atoms
tau = 1.0;            # mean lifetime - set to 1, i.e., measure time in units of tau
tmax = 5.0 * tau;     # integrate from tau = 0 to 5

# Defning different dt/tau values to be used as specified by the lab manual
dt_over_tau_list = [0.05, 0.2, 0.8, 1.0, 1.5];

method_list = ["Euler", "RK2", "RK4"];

# ODE: dN/dt = -N/tau
def f(t, N):
    return (-N / tau);

# One integrator for ALL methods (final step adjusted to hit tmax exactly)
def integrator (method, dt_over_tau):

    dt = dt_over_tau * tau;

    times_list = [0.0];

    # Add full dt steps as long as the next step would not overshoot tmax
    while times_list[-1] + dt < tmax:
        times_list.append(times_list[-1] + dt);

    if times_list[-1] < tmax:
        times_list.append(tmax);

    times = np.array(times_list);
    steps = len(times) - 1; # If there are N time points, there are N-1 update steps

    N = np.zeros(steps + 1);
    N[0] = N0;

    # Begin iterating through N time points
    for i in range(steps):

        t_i = times[i];
        h = times[i + 1] - times[i];
        y = N[i];

        if method == "Euler":
            N[i + 1] = y + h * f(t_i, y);

        elif method == "RK2":
            k1 = f(t_i, y);
            y_mid = y + 0.5 * h * k1;
            k2 = f(t_i + 0.5 * h, y_mid);
            N[i + 1] = y + h * k2;

        elif method == "RK4":
            k1 = f(t_i, y);
            k2 = f(t_i + 0.5*h, y + 0.5*h*k1);
            k3 = f(t_i + 0.5*h, y + 0.5*h*k2);
            k4 = f(t_i + h,     y + h*k3);
            N[i + 1] = y + (h / 6.0) * (k1 + 2.0*k2 + 2.0*k3 + k4);

    return times, N;

# First generating exact solution
t_exact = np.linspace(0.0, tmax, 1000);
N_exact = np.array([N0 * exp(-t / tau) for t in t_exact]);

# Precompute results for all methods + all dt values so we don't compute integrator repeatedly when plotting.
results = [];

for method in method_list:

    method_results = [];

    for dt_over_tau in dt_over_tau_list:

        # Numerical approximation
        times, N = integrator(method, dt_over_tau);

        #Exact solution evaluated at the SAME time points as the numerical method
        exact = np.array([N0 * exp(-t / tau) for t in times]);

        # "Global" deviation at each time point: numerical minus exact
        error = N - exact;
        abs_error = np.abs(error);
        frac_error = abs_error / np.abs(exact);

        method_results.append([times, N, exact, abs_error, frac_error]);

    results.append(method_results);

# For each dt/tau, ploting exact + Euler + RK2 + RK4

for k in range(len(dt_over_tau_list)):

    dt_over_tau = dt_over_tau_list[k];

    plt.figure();
    plt.xlabel('t / tau');
    plt.ylabel('N(t)');
    plt.title(f'Approximations vs exact (dt/tau = {dt_over_tau:g})');

    plt.plot(t_exact / tau, N_exact, label = "exact", linewidth = 2);

    # Numerical solutions - markers show the actual time steps used
    for m in range(len(method_list)):
        method = method_list[m];
        times = results[m][k][0];
        N  = results[m][k][1];
        plt.plot(times / tau, N, marker = 'o', markersize = 3, label = method);

    plt.legend();
    plt.grid(True, alpha = 0.3);
    plt.show();


# Error vs time (absolute + fractional) for RK2 and RK4

for m in range(len(method_list)):

    method = method_list[m];

    # Previously done Euler
    if method == "Euler":
        continue;

    # Absolute error vs time
    plt.figure();
    plt.xlabel('t / tau');
    plt.ylabel(r'|N - N_exact|');
    plt.title(f'Global deviation vs time (absolute) : {method}');

    for k in range(len(dt_over_tau_list)):
        dt_over_tau = dt_over_tau_list[k];
        times = results[m][k][0];
        abs_error = results[m][k][3];
        plt.plot(times / tau, abs_error, marker = 'o', markersize = 3, label = f"dt/tau = {dt_over_tau:g}");

    plt.legend();
    plt.grid(True, alpha = 0.3);
    plt.show();

    # Fractional error vs time
    plt.figure();
    plt.xlabel('t / tau');
    plt.ylabel(r'|N - N_exact| / |N_exact|');
    plt.title(f'Global deviation vs time (fractional) : {method}');

    for k in range(len(dt_over_tau_list)):
        dt_over_tau = dt_over_tau_list[k];
        times = results[m][k][0];
        frac_error = results[m][k][4];
        plt.plot(times / tau, frac_error, marker = 'o', markersize = 3, label = f"dt/tau = {dt_over_tau:g}");

    plt.legend();
    plt.grid(True, alpha = 0.3);
    plt.yscale('log');
    plt.show();


# Error vs step size at fixed time t = 5 tau and comparing methods
plt.figure();
plt.xlabel('dt / tau');
plt.ylabel(r'|error| at t = 5 tau');
plt.title('Step-size dependence at fixed time (absolute)');

for m in range(len(method_list)):
    abs_at_tmax = [];
    for k in range(len(dt_over_tau_list)):
        abs_at_tmax.append(results[m][k][3][-1]);
    plt.plot(dt_over_tau_list, abs_at_tmax, marker = 'o', label = method_list[m]);

plt.legend();
plt.grid(True, alpha = 0.3);
plt.show();


plt.figure();
plt.xlabel('dt / tau');
plt.ylabel(r'|error| / |exact| at t = 5 tau');
plt.title('Step-size dependence at fixed time (fractional)');

for m in range(len(method_list)):
    frac_at_tmax = [];
    for k in range(len(dt_over_tau_list)):
        frac_at_tmax.append(results[m][k][4][-1]);
    plt.plot(dt_over_tau_list, frac_at_tmax, marker = 'o', label = method_list[m]);

plt.legend();
plt.grid(True, alpha = 0.3);
plt.show();

#EOF


